In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Tugas7") \
    .master("local[*]") \
    .getOrCreate()

26/09/30 23:25:58 WARN Utils: Your hostname, cahyoalim-ThinkPad-L14-Gen-2 resolves to a loopback address: 127.0.1.1; using 192.168.1.194 instead (on interface wlp9s0)
26/09/30 23:25:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 23:25:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/30 23:25:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


**A. EXTRACT** *(bobot 10%)*

Baca ketiga sumber data **dari HDFS** (bukan disk lokal) menjadi tiga Spark DataFrame.


In [17]:
#soal A
cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)
transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)

In [19]:
cabang.show()
menu.show()
transaksi.show()

+---------+--------------+----------+--------------------+
|cabang_id|kepala_barista|      kota|         nama_cabang|
+---------+--------------+----------+--------------------+
|        1|         Nadia|  Magelang|Kopi Nusantara Ma...|
|        2|          Reza|Yogyakarta|Kopi Nusantara Yo...|
|        3|         Nadia|  Semarang|Kopi Nusantara Se...|
+---------+--------------+----------+--------------------+

+-------+---------+--------------+-----+
|menu_id|nama_menu| kategori_menu|harga|
+-------+---------+--------------+-----+
|      1|   Menu-1|Makanan Ringan|35000|
|      2|   Menu-2|Makanan Ringan|22000|
|      3|   Menu-3|Makanan Ringan|30000|
|      4|   Menu-4|      Non-Kopi|22000|
|      5|   Menu-5|          Kopi|22000|
|      6|   Menu-6|Makanan Ringan|22000|
|      7|   Menu-7|          Kopi|18000|
|      8|   Menu-8|      Non-Kopi|30000|
|      9|   Menu-9|Makanan Ringan|35000|
|     10|  Menu-10|          Kopi|18000|
|     11|  Menu-11|          Kopi|22000|
|     12|  M

**B. TRANSFORM — Join & Kolom Turunan** *(bobot 20%)*

Gabungkan ketiga tabel, tambahkan kolom `total_penjualan` (`qty x harga`).


In [26]:
#Soal B

a = transaksi.join(cabang, on="cabang_id", how="inner")
a = a.join(menu, on="menu_id", how="inner")

a = a.withColumn("total_penjualan", col("qty") * col("harga"))
a.show()

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


**C. TRANSFORM — Window Function** *(bobot 20%)*

Tentukan **top-2 menu terlaris** (berdasarkan `total_penjualan`) di **setiap cabang**, gunakan `row_number()`.

In [27]:
#Soal C
menu = a.groupBy("nama_menu", "nama_cabang").agg(spark_sum("total_penjualan").alias("total"))

w = Window.partitionBy("nama_cabang").orderBy(col("total").desc())

top_menu = menu.withColumn("top-2 menu terlaris", row_number().over(w)).filter(col("top-2 menu terlaris") <= 2)

top_menu.show()

+---------+--------------------+-------+-------------------+
|nama_menu|         nama_cabang|  total|top-2 menu terlaris|
+---------+--------------------+-------+-------------------+
|   Menu-1|Kopi Nusantara Ma...|8505000|                  1|
|  Menu-12|Kopi Nusantara Ma...|8085000|                  2|
|  Menu-13|Kopi Nusantara Se...|8750000|                  1|
|   Menu-9|Kopi Nusantara Se...|8715000|                  2|
|  Menu-12|Kopi Nusantara Yo...|9275000|                  1|
|  Menu-15|Kopi Nusantara Yo...|8160000|                  2|
+---------+--------------------+-------+-------------------+



**D. TRANSFORM — Spark SQL** *(bobot 20%)*

Menggunakan **Spark SQL murni**, tampilkan total penjualan per `kepala_barista`, diurutkan dari tertinggi.

In [32]:
#Soal D

a.createOrReplaceTempView("data")
D = spark.sql('''
select kepala_barista, sum(total_penjualan) as total_penjualan from data group by kepala_barista order by total_penjualan desc''')
D.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



**E. LOAD** *(bobot 15%)*

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path `/user/[username]/tugas7/processed/laporan`, dipartisi berdasarkan `kategori_menu`. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

In [41]:
# Soal E

a.write.mode("overwrite").partitionBy("kategori_menu").parquet("hdfs://localhost:9000/user/cahyoalim/tugas7/processed/laporan")

baca = spark.read.parquet("hdfs://localhost:9000/user/cahyoalim/tugas7/processed/laporan")


baca.show()
baca.count()

+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu|harga|total_penjualan| kategori_menu|
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|35000|         140000|Makanan Ringan|
|      9|        3|   KN8|  4|         Nadia|  Semarang|Kopi Nusantara Se...|   Menu-9|35000|         140000|Makanan Ringan|
|      3|        1|  KN10|  1|         Nadia|  Magelang|Kopi Nusantara Ma...|   Menu-3|30000|          30000|Makanan Ringan|
|     13|        2|  KN13|  3|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-13|35000|         105000|Makanan Ringan|
|     12|        3|  KN15|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-12|35000|          35000|Makanan Ringan|


4000

**F. Ringkasan Eksekutif & Rekomendasi** *(bobot 15%)*

Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (**minimal 120 kata**): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

In [45]:
#Soal F

ringkasan = a.groupBy("nama_cabang").agg(spark_sum("total_penjualan").alias("total_penjualan"), 
                                         count("trx_id").alias("jumlah_transaksi"), avg("total_penjualan").alias("rata_rata")).orderBy(col("jumlah_transaksi").desc())
ringkasan.show()

+--------------------+---------------+----------------+-----------------+
|         nama_cabang|total_penjualan|jumlah_transaksi|        rata_rata|
+--------------------+---------------+----------------+-----------------+
|Kopi Nusantara Se...|       90542000|            1393| 64997.8463747308|
|Kopi Nusantara Yo...|       84645000|            1305|64862.06896551724|
|Kopi Nusantara Ma...|       87138000|            1302|66926.26728110599|
+--------------------+---------------+----------------+-----------------+



**kesimpulan**

Berdasarkan analisis tabel ringkasan eksekutif, cabang dengan kinerja terbaik adalah Kopi Nusantara Semarang dengan total penjualan mencapai Rp90.542.000 dari 1.393 transaksi. Hasil ini menunjukkan kunjungan dan daya beli pelanggan paling tinggi dibandingkan cabang lainnya. Sebaliknya, cabang dengan kinerja terlemah adalah Kopi Nusantara Yogyakarta yang mencatatkan total penjualan terendah yaitu Rp84.645.000 dari 1.305 transaksi.   Untuk mendongkrak performa Kopi Nusantara Yogyakarta, manajemen disarankan mempromosikan lebih gencar menu unggulannya berdasarkan temuan bagian C, yaitu Menu-12 dan Menu-15. Meskipun kedua menu tersebut menjadi produk terlaris di Yogyakarta, total omzet cabang masih tertinggal. Strategi promosi seperti penawaran paket bundling minuman dengan snack dan pemasaran digital fokus pada Menu-12 dan Menu-15 diharapkan dapat menarik lebih banyak pelanggan dan meningkatkan rata-rata nilai transaksi.   